In [51]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [52]:
#Feature
def generate_batch(batch_size, S, n):
    input1 = torch.rand(batch_size, n)
    input2 = torch.rand(batch_size, n)
    dataset = input1 * torch.gt(input2, S).float()
    return dataset

In [53]:
# Neural Network Model
class NeuralNetwork(nn.Module):
    def __init__(self, n, m, activation):
        super(NeuralNetwork, self).__init__()
        self.n = n
        self.m = m
        self.weight = nn.init.xavier_normal_(torch.rand(m, n))
        self.bias = torch.zeros(n)
        self.activation = activation
        self.weight = nn.Parameter(self.weight)
        self.bias = nn.Parameter(self.bias)


    def forward(self, x):
        h = torch.matmul(x, self.weight.t())
        x_prime = self.activation(torch.matmul(h, self.weight) + self.bias)
        return x_prime
    

In [ ]:
n = 5 # number of input features
m = 2 # number of hidden units
importance = torch.arange(0, n).float()  # importance of each feature
importance = torch.pow(0.7, importance)  # apply a power transformation to the importance values
def loss_fn (x, x_prime, importance):
    return torch.sum(importance * (x - x_prime) ** 2, dim=1).mean()
# m < n to ensure that the model is underparameterized
model = NeuralNetwork(n, m, nn.ReLU())
dataset = generate_batch(3, 0, n)

torch.Size([3, 5])
